# Trading Activity Patterns

## Time-Series Analysis of US Interest Rate Swap Trading

This notebook provides comprehensive analysis of trading activity patterns:

1. **Intraday Patterns** - Hourly and minute-level trading activity
2. **Daily Patterns** - Day-of-week effects
3. **Volume Clustering** - Trade clustering and burst detection
4. **Event Analysis** - FOMC and data release impacts
5. **Seasonality** - Month-end, quarter-end effects
6. **Correlation Analysis** - Cross-tenor activity correlation

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
from scipy import stats

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder

cache_path = r"/tmp/sdr_cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

## 1. Data Loading (Multiple Days)

In [ ]:
# Fetch multiple days for pattern analysis
start = NY_tz.localize(datetime.datetime(2025, 12, 9, 6, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 19, 18, 0))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start,
    end_timestamp=end,
    agency="CFTC",
    asset_class="RATES"
)

print(f"Total trades fetched: {len(raw_df):,}")
print(f"Date range: {start.date()} to {end.date()}")

In [ ]:
def preprocess_pattern_data(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess data for pattern analysis."""
    df = df.copy()
    
    # Filter to new trades
    df = df[df['Action type'] == 'NEWT'].copy()
    
    # Filter USD SOFR
    df = df[df['Notional currency-Leg 1'] == 'USD'].copy()
    sofr_mask = df['UPI Underlier Name'].str.contains('SOFR', case=False, na=False)
    df = df[sofr_mask].copy()
    
    # Parse timestamps
    df['Event timestamp'] = pd.to_datetime(df['Event timestamp'], utc=True)
    df['Execution Timestamp'] = pd.to_datetime(df['Execution Timestamp'], utc=True)
    df['Event_Time_NY'] = df['Event timestamp'].dt.tz_convert('America/New_York')
    
    # Extract time components
    df['Date'] = df['Event_Time_NY'].dt.date
    df['Hour'] = df['Event_Time_NY'].dt.hour
    df['Minute'] = df['Event_Time_NY'].dt.minute
    df['Day_of_Week'] = df['Event_Time_NY'].dt.dayofweek
    df['Day_Name'] = df['Event_Time_NY'].dt.day_name()
    df['Week'] = df['Event_Time_NY'].dt.isocalendar().week
    df['Month'] = df['Event_Time_NY'].dt.month
    df['Time_Bucket_5min'] = df['Event_Time_NY'].dt.floor('5min')
    df['Time_Bucket_15min'] = df['Event_Time_NY'].dt.floor('15min')
    df['Time_Bucket_30min'] = df['Event_Time_NY'].dt.floor('30min')
    df['Time_Bucket_1H'] = df['Event_Time_NY'].dt.floor('H')
    
    # Parse notional
    df['Notional amount-Leg 1'] = df['Notional amount-Leg 1'].astype(str).str.replace(',', '')
    df['Notional amount-Leg 1'] = pd.to_numeric(df['Notional amount-Leg 1'], errors='coerce')
    
    # Parse dates for tenor
    df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
    df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')
    df['Tenor_Days'] = (df['Expiration Date'] - df['Effective Date']).dt.days
    df['Tenor_Years'] = df['Tenor_Days'] / 365.25
    
    # Tenor buckets
    def tenor_bucket(years):
        if pd.isna(years) or years <= 0:
            return 'Unknown'
        elif years <= 2:
            return 'Front End (0-2Y)'
        elif years <= 5:
            return 'Belly (2-5Y)'
        elif years <= 10:
            return 'Intermediate (5-10Y)'
        else:
            return 'Long End (10Y+)'
    
    df['Tenor_Bucket'] = df['Tenor_Years'].apply(tenor_bucket)
    
    return df.reset_index(drop=True)

df = preprocess_pattern_data(raw_df)
print(f"Processed USD SOFR trades: {len(df):,}")

## 2. Intraday Trading Patterns

In [ ]:
# Hourly trade distribution (average across days)
hourly_by_day = df.groupby(['Date', 'Hour'])['Dissemination Identifier'].count().reset_index()
hourly_by_day.columns = ['Date', 'Hour', 'Trade_Count']

hourly_avg = hourly_by_day.groupby('Hour')['Trade_Count'].agg(['mean', 'std', 'min', 'max'])
hourly_avg.columns = ['Mean', 'Std', 'Min', 'Max']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=hourly_avg.index,
    y=hourly_avg['Mean'],
    error_y=dict(type='data', array=hourly_avg['Std']),
    marker_color='steelblue',
    name='Mean +/- StdDev'
))

fig.update_layout(
    title='Average Hourly Trading Activity (with variability)',
    xaxis_title='Hour (NY Time)',
    yaxis_title='Average Trade Count',
    height=500
)
fig.show()

In [ ]:
# 5-minute resolution intraday profile
df['Time_of_Day'] = df['Event_Time_NY'].dt.hour + df['Event_Time_NY'].dt.minute / 60

time_bins = np.arange(6, 18.5, 5/60)  # 5-minute bins from 6 AM to 6 PM

intraday_profile = df.groupby(pd.cut(df['Time_of_Day'], bins=time_bins)).size()
intraday_profile = intraday_profile / len(df['Date'].unique())  # Average per day

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[b.mid for b in intraday_profile.index],
    y=intraday_profile.values,
    mode='lines',
    fill='tozeroy',
    fillcolor='rgba(70, 130, 180, 0.3)',
    line=dict(color='steelblue', width=2)
))

# Add key time markers
key_times = [
    (7, 'CME Open'),
    (8.5, '8:30 Data'),
    (9.5, 'NYSE Open'),
    (10, '10AM Data'),
    (14, 'FOMC'),
    (16, 'NYSE Close'),
    (17, 'CME Close')
]
for time, label in key_times:
    fig.add_vline(x=time, line_dash='dash', line_color='gray', opacity=0.5)
    fig.add_annotation(x=time, y=intraday_profile.max() * 1.1, text=label,
                       showarrow=False, textangle=-45, font_size=10)

fig.update_layout(
    title='Intraday Trading Profile (5-minute resolution)',
    xaxis_title='Time of Day (NY)',
    yaxis_title='Average Trades per 5 minutes',
    height=500
)
fig.update_xaxes(tickvals=[6,7,8,9,10,11,12,13,14,15,16,17,18],
                 ticktext=['6AM','7AM','8AM','9AM','10AM','11AM','12PM','1PM','2PM','3PM','4PM','5PM','6PM'])
fig.show()

In [ ]:
# Intraday pattern by tenor
fig = go.Figure()

colors = {'Front End (0-2Y)': 'blue', 'Belly (2-5Y)': 'green', 
          'Intermediate (5-10Y)': 'orange', 'Long End (10Y+)': 'red'}

for tenor in ['Front End (0-2Y)', 'Belly (2-5Y)', 'Intermediate (5-10Y)', 'Long End (10Y+)']:
    tenor_df = df[df['Tenor_Bucket'] == tenor]
    if len(tenor_df) > 0:
        hourly = tenor_df.groupby('Hour').size()
        hourly = hourly / len(df['Date'].unique())
        fig.add_trace(go.Scatter(
            x=hourly.index,
            y=hourly.values,
            mode='lines+markers',
            name=tenor,
            line=dict(color=colors.get(tenor, 'gray'))
        ))

fig.update_layout(
    title='Intraday Trading Pattern by Tenor',
    xaxis_title='Hour (NY Time)',
    yaxis_title='Average Trade Count',
    height=500,
    legend=dict(x=0.02, y=0.98)
)
fig.show()

## 3. Day-of-Week Patterns

In [ ]:
# Day of week distribution
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

daily_trades = df.groupby('Date').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
daily_trades.columns = ['Date', 'Trade_Count', 'Total_Notional']
daily_trades['Day_Name'] = pd.to_datetime(daily_trades['Date']).dt.day_name()

day_stats = daily_trades.groupby('Day_Name').agg({
    'Trade_Count': ['mean', 'std', 'count'],
    'Total_Notional': ['mean', 'std']
}).round(0)
day_stats.columns = ['Avg_Trades', 'Trades_Std', 'Days_Count', 'Avg_Notional', 'Notional_Std']
day_stats = day_stats.reindex([d for d in day_order if d in day_stats.index])

print("Day-of-Week Trading Statistics:")
display(day_stats)

In [ ]:
# Visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=['Average Trade Count', 'Average Notional ($B)'])

fig.add_trace(
    go.Bar(x=day_stats.index, y=day_stats['Avg_Trades'],
           error_y=dict(type='data', array=day_stats['Trades_Std']),
           marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=day_stats.index, y=day_stats['Avg_Notional']/1e9,
           error_y=dict(type='data', array=day_stats['Notional_Std']/1e9),
           marker_color='darkgreen'),
    row=1, col=2
)

fig.update_layout(title_text='Day-of-Week Trading Patterns', height=400, showlegend=False)
fig.show()

In [ ]:
# Heatmap: Day of Week vs Hour
dow_hour = df.groupby(['Day_Name', 'Hour']).size().reset_index(name='Count')
dow_hour_pivot = dow_hour.pivot(index='Day_Name', columns='Hour', values='Count').fillna(0)
dow_hour_pivot = dow_hour_pivot.reindex([d for d in day_order if d in dow_hour_pivot.index])

fig = go.Figure(data=go.Heatmap(
    z=dow_hour_pivot.values,
    x=dow_hour_pivot.columns,
    y=dow_hour_pivot.index,
    colorscale='Blues',
    hoverongaps=False
))

fig.update_layout(
    title='Trading Activity Heatmap: Day of Week vs Hour',
    xaxis_title='Hour (NY Time)',
    yaxis_title='Day of Week',
    height=400
)
fig.show()

## 4. Trade Clustering Analysis

In [ ]:
# Inter-trade time analysis
df_sorted = df.sort_values('Event timestamp')
df_sorted['Inter_Trade_Time'] = df_sorted['Event timestamp'].diff().dt.total_seconds()

# Filter to same-day trades
df_sorted['Date_Diff'] = df_sorted['Date'] != df_sorted['Date'].shift(1)
df_sorted.loc[df_sorted['Date_Diff'], 'Inter_Trade_Time'] = np.nan

inter_trade = df_sorted['Inter_Trade_Time'].dropna()
inter_trade = inter_trade[(inter_trade > 0) & (inter_trade < 600)]  # Cap at 10 min

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=inter_trade,
    nbinsx=60,
    marker_color='steelblue'
))

fig.add_vline(x=inter_trade.median(), line_dash='dash', line_color='red',
              annotation_text=f'Median: {inter_trade.median():.1f}s')

fig.update_layout(
    title='Inter-Trade Time Distribution (capped at 10 min)',
    xaxis_title='Seconds Between Trades',
    yaxis_title='Count',
    height=400
)
fig.show()

In [ ]:
# Identify trading bursts (clusters of trades)
# Define a burst as 5+ trades within 60 seconds

def identify_bursts(df, time_col='Event timestamp', window_seconds=60, min_trades=5):
    """
    Identify trading bursts - periods of intense activity.
    """
    df = df.sort_values(time_col).copy()
    
    # Rolling count of trades in the window
    df['time_int'] = df[time_col].astype('int64') // 1e9  # Convert to seconds
    
    bursts = []
    i = 0
    while i < len(df):
        window_end = df.iloc[i]['time_int'] + window_seconds
        window_trades = df[(df['time_int'] >= df.iloc[i]['time_int']) & 
                           (df['time_int'] <= window_end)]
        
        if len(window_trades) >= min_trades:
            burst = {
                'start_time': window_trades[time_col].min(),
                'end_time': window_trades[time_col].max(),
                'trade_count': len(window_trades),
                'total_notional': window_trades['Notional amount-Leg 1'].sum()
            }
            bursts.append(burst)
            # Skip to end of burst
            i = window_trades.index[-1] + 1 if isinstance(window_trades.index[-1], int) else len(df)
            i = df.index.get_loc(window_trades.index[-1]) + 1 if window_trades.index[-1] in df.index else len(df)
        else:
            i += 1
    
    return pd.DataFrame(bursts)

bursts_df = identify_bursts(df, window_seconds=60, min_trades=5)
print(f"Identified {len(bursts_df)} trading bursts (5+ trades in 60 seconds)")

if len(bursts_df) > 0:
    print(f"\nBurst Statistics:")
    print(f"  Average trades per burst: {bursts_df['trade_count'].mean():.1f}")
    print(f"  Max trades in burst: {bursts_df['trade_count'].max()}")
    print(f"  Average notional per burst: ${bursts_df['total_notional'].mean():,.0f}")

In [ ]:
# Volume profile: cumulative trades through the day
# Pick a sample day
sample_date = df['Date'].value_counts().idxmax()
sample_df = df[df['Date'] == sample_date].sort_values('Event_Time_NY')

sample_df['Cumulative_Trades'] = range(1, len(sample_df) + 1)
sample_df['Cumulative_Notional'] = sample_df['Notional amount-Leg 1'].cumsum()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Cumulative Trade Count', 'Cumulative Notional ($B)'])

fig.add_trace(
    go.Scatter(x=sample_df['Event_Time_NY'], y=sample_df['Cumulative_Trades'],
               mode='lines', line=dict(color='steelblue', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=sample_df['Event_Time_NY'], y=sample_df['Cumulative_Notional']/1e9,
               mode='lines', line=dict(color='darkgreen', width=2)),
    row=2, col=1
)

fig.update_layout(title_text=f'Cumulative Trading Activity - {sample_date}', height=600)
fig.show()

## 5. Cross-Tenor Activity Correlation

In [ ]:
# Trade count correlation between tenor buckets
hourly_tenor = df.groupby(['Date', 'Hour', 'Tenor_Bucket']).size().unstack(fill_value=0)

# Calculate correlation matrix
tenor_corr = hourly_tenor.corr()

fig = go.Figure(data=go.Heatmap(
    z=tenor_corr.values,
    x=tenor_corr.columns,
    y=tenor_corr.index,
    colorscale='RdBu_r',
    zmid=0,
    text=np.round(tenor_corr.values, 2),
    texttemplate='%{text}',
    textfont={'size': 12}
))

fig.update_layout(
    title='Cross-Tenor Trading Activity Correlation',
    height=500,
    width=600
)
fig.show()

In [ ]:
# Daily tenor mix
daily_tenor = df.groupby(['Date', 'Tenor_Bucket']).size().unstack(fill_value=0)
daily_tenor_pct = daily_tenor.div(daily_tenor.sum(axis=1), axis=0) * 100

fig = go.Figure()

for tenor in ['Front End (0-2Y)', 'Belly (2-5Y)', 'Intermediate (5-10Y)', 'Long End (10Y+)']:
    if tenor in daily_tenor_pct.columns:
        fig.add_trace(go.Scatter(
            x=daily_tenor_pct.index,
            y=daily_tenor_pct[tenor],
            mode='lines+markers',
            name=tenor,
            stackgroup='one'
        ))

fig.update_layout(
    title='Daily Tenor Mix (% of trades)',
    xaxis_title='Date',
    yaxis_title='Share (%)',
    height=500,
    yaxis=dict(range=[0, 100])
)
fig.show()

## 6. Volume-Weighted Time of Day

In [ ]:
# VWAP-style analysis: Volume-weighted average time of trading
# Calculate the "center of mass" of trading activity

for date in df['Date'].unique():
    day_df = df[df['Date'] == date].copy()
    if len(day_df) > 0:
        day_df['Time_Minutes'] = day_df['Hour'] * 60 + day_df['Minute']
        
        # Trade-weighted
        trade_weighted_time = day_df['Time_Minutes'].mean()
        
        # Notional-weighted
        notional = day_df['Notional amount-Leg 1'].fillna(0)
        if notional.sum() > 0:
            notional_weighted_time = (day_df['Time_Minutes'] * notional).sum() / notional.sum()
        else:
            notional_weighted_time = trade_weighted_time

# Calculate for the full dataset
df['Time_Minutes'] = df['Hour'] * 60 + df['Minute']
notional = df['Notional amount-Leg 1'].fillna(0)

trade_weighted = df['Time_Minutes'].mean()
notional_weighted = (df['Time_Minutes'] * notional).sum() / notional.sum() if notional.sum() > 0 else trade_weighted

print("Volume-Weighted Time of Day Analysis:")
print("="*50)
print(f"Trade-weighted average time: {int(trade_weighted//60):02d}:{int(trade_weighted%60):02d} NY")
print(f"Notional-weighted average time: {int(notional_weighted//60):02d}:{int(notional_weighted%60):02d} NY")

## 7. Daily Activity Summary

In [ ]:
# Daily activity time series
daily_summary = df.groupby('Date').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': ['sum', 'mean'],
    'Time_Minutes': 'std'  # Trading activity spread
}).reset_index()
daily_summary.columns = ['Date', 'Trade_Count', 'Total_Notional', 'Avg_Trade_Size', 'Time_Spread']

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['Daily Trade Count', 'Daily Notional ($B)', 'Average Trade Size ($M)'])

fig.add_trace(
    go.Bar(x=daily_summary['Date'], y=daily_summary['Trade_Count'], marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=daily_summary['Date'], y=daily_summary['Total_Notional']/1e9, marker_color='darkgreen'),
    row=2, col=1
)

fig.add_trace(
    go.Bar(x=daily_summary['Date'], y=daily_summary['Avg_Trade_Size']/1e6, marker_color='purple'),
    row=3, col=1
)

fig.update_layout(title_text='Daily Trading Summary', height=700, showlegend=False)
fig.show()

## 8. Summary Report

In [ ]:
print("="*70)
print("TRADING PATTERN SUMMARY")
print("="*70)
print(f"\nAnalysis Period: {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
print(f"Trading Days Analyzed: {len(df['Date'].unique())}")
print(f"Total Trades: {len(df):,}")

print(f"\nIntraday Patterns:")
peak_hour = hourly_avg['Mean'].idxmax()
print(f"  Peak Trading Hour: {peak_hour}:00 - {peak_hour+1}:00 NY ({hourly_avg.loc[peak_hour, 'Mean']:.0f} avg trades)")
print(f"  Trade-weighted Avg Time: {int(trade_weighted//60):02d}:{int(trade_weighted%60):02d} NY")

print(f"\nDay-of-Week Patterns:")
busiest_day = day_stats['Avg_Trades'].idxmax()
slowest_day = day_stats['Avg_Trades'].idxmin()
print(f"  Busiest Day: {busiest_day} ({day_stats.loc[busiest_day, 'Avg_Trades']:.0f} avg trades)")
print(f"  Slowest Day: {slowest_day} ({day_stats.loc[slowest_day, 'Avg_Trades']:.0f} avg trades)")

print(f"\nTrade Clustering:")
print(f"  Median Inter-Trade Time: {inter_trade.median():.1f} seconds")
print(f"  Trading Bursts Identified: {len(bursts_df)}" if len(bursts_df) > 0 else "  Trading Bursts Identified: 0")